In [ ]:

import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()


In [23]:
# CONNECT TO QDRANT
import os
import json
import qdrant_client
from qdrant_client.http.models import Filter, FieldCondition, MatchValue,  MatchAny, PointStruct, VectorParams, Distance
from qdrant_client import QdrantClient


ENDLESSFORMS_QDRANT_URL = os.getenv("ENDLESSFORMS_QDRANT_URL")
ENDLESSFORMS_TEST_CLUSTER_KEY = os.getenv("ENDLESSFORMS_TEST_CLUSTER_KEY")

qdrantclient = QdrantClient(
    url=ENDLESSFORMS_QDRANT_URL,
    api_key=ENDLESSFORMS_TEST_CLUSTER_KEY,
)

print(qdrantclient)

collections = qdrantclient.get_collections()
list(collections)[0][1]

[CollectionDescription(name='TEXAS_PIPE_EXP'),
 CollectionDescription(name='test_intfloat_e5-base-v2'),
 CollectionDescription(name='MATTSCO-VOYAGEAI_LARGE_2048'),
 CollectionDescription(name='ENCODER-TEST-INTFLOAT-MULTILINGUAL-E5-BASE'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_intfloat_e5-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-26_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='DENSE_VECTOR_FERGUSON_DEC_26_OPENAI_LARGE'),
 CollectionDescription(name='FERGUSON_VOYAGEAI_LARGE_2048_JAN1'),
 CollectionDescription(name='test_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-20_sentence-transformers_all-mpnet-base-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='test_sentence-transformers_all-MiniLM-L6-v2'),
 CollectionDescription(name='ENCODER_TEST_2024-12-25_BAAI_bge-reranker-v2-m3'),
 CollectionDescription(na

In [22]:
# #DELETE OLD COLLECTION
COLLECTION_NAME = "ENCODER-TEST-MULTILINGUALE5BASE"

collection_name = COLLECTION_NAME

# Delete the collection
response = qdrantclient.delete_collection(collection_name=collection_name)

# Print the response
print(response)

True


In [20]:
import os
from qdrant_client import QdrantClient
from tabulate import tabulate
from typing import Dict, List
import humanize

def get_collection_stats(client: QdrantClient) -> List[Dict]:
    """
    Get statistics for all collections in a Qdrant environment.
    
    Args:
        client: QdrantClient instance
        
    Returns:
        List of dictionaries containing collection statistics
    """
    stats = []
    collections = client.get_collections().collections
    
    for collection in collections:
        collection_name = collection.name
        
        # Get detailed collection info
        collection_info = client.get_collection(collection_name)
        
        # Get collection telemetry for storage size
        telemetry = client._collection_telemetry(collection_name)
        storage_size = 0
        
        # Extract storage size from telemetry
        if telemetry and 'collections_collections_store_size' in telemetry:
            storage_size = telemetry['collections_collections_store_size']
        elif telemetry and 'disk_space_used' in telemetry:
            storage_size = telemetry['disk_space_used']
            
        # Get collection statistics
        collection_stats = {
            'name': collection_name,
            'vectors_count': collection_info.points_count,
            'segments_count': collection_info.segments_count,
            'status': collection_info.status,
            'storage_size': storage_size,
            'vector_size': collection_info.config.params.vectors.size,
            'distance': collection_info.config.params.vectors.distance
        }
        
        stats.append(collection_stats)
    
    return stats

def format_stats(stats: List[Dict]) -> str:
    """
    Format collection statistics into a human-readable table.
    
    Args:
        stats: List of collection statistics dictionaries
        
    Returns:
        Formatted table string
    """
    table_data = []
    
    for stat in stats:
        table_data.append([
            stat['name'],
            f"{stat['vectors_count']:,}",
            stat['segments_count'],
            stat['status'],
            humanize.naturalsize(stat['storage_size'], binary=True),
            stat['vector_size'],
            stat['distance']
        ])
    
    headers = ['Collection', 'Vectors', 'Segments', 'Status', 'Storage Size', 'Vector Size', 'Distance']
    return tabulate(table_data, headers=headers, tablefmt='grid')


        

        


In [49]:

# Get and format statistics
stats = get_collection_stats(qdrantclient)
formatted_stats = format_stats(stats)

# Print results
print("\nQdrant Collections Statistics:")
print(formatted_stats)

# Print summary
total_vectors = sum(stat['vectors_count'] for stat in stats)
total_storage = sum(stat['storage_size'] for stat in stats)

print(f"\nTotal Collections: {len(stats)}")
print(f"Total Vectors: {total_vectors:,}")
print(f"Total Storage Size: {humanize.naturalsize(total_storage, binary=True)}")



Qdrant Collections Statistics:
+-----------------------------------------------------------------+-----------+------------+----------+----------------+
| Collection                                                      | Vectors   |   Segments | Status   | Storage Size   |
+=================================================================+===========+============+==========+================+
| TEXAS_PIPE_EXP                                                  | 266,992   |          6 | green    | 0 Bytes        |
+-----------------------------------------------------------------+-----------+------------+----------+----------------+
| test_intfloat_e5-base-v2                                        | 100       |          2 | green    | 0 Bytes        |
+-----------------------------------------------------------------+-----------+------------+----------+----------------+
| MATTSCO-VOYAGEAI_LARGE_2048                                     | 22,659    |          2 | green    | 0 Bytes        |
